# SAE Zero-Ablation Unlearning — LLaVA-Next (llama3-llava-next-8b-hf)

**What this notebook does:**
1. Loads pre-built dataset images from `data/` + `probe_images/` (exported by the baseline notebook).
2. Loads the LLaVA-Next model in **4-bit** (fits on a single Kaggle T4).
3. Loads the pre-trained **TopK SAE** from `/kaggle/input/models/akgiiith/vision-sae/pytorch/default/1/vision_sae_final.pt`.
4. Runs **feature discovery** — finds the SAE latents most activated by zebra images.
5. Installs a **forward hook** on the vision encoder's last layer that:
   - Encodes hidden states through the SAE,
   - **Zeroes out** the identified zebra features (zero ablation),
   - Reconstructs and passes the modified states onward.
6. Evaluates the ablated model on the same **FA / RA / LL / AR** metrics as the baseline.
7. Prints a side-by-side comparison and saves results to JSON.

**OOM-safe design (Kaggle Free T4 — 16 GB VRAM):**
- Model loaded in 4-bit NF4 quantization via BitsAndBytes.
- SAE stays in `float16` on the same device as the vision tower.
- Feature discovery processes one image at a time (no batch accumulation in VRAM).
- Dataset images are streamed from disk, not held in RAM.
- `torch.cuda.empty_cache()` + `gc.collect()` called at every major boundary.

In [ ]:
!pip install -q datasets transformers accelerate bitsandbytes pillow tqdm

In [ ]:
import os, json, gc, math, time, random
from pathlib import Path
from PIL import Image, ImageFilter

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

# ── Paths ────────────────────────────────────────────────────────────────
MODEL_ID  = "llava-hf/llama3-llava-next-8b-hf"
SAE_PATH  = "/kaggle/input/models/akgiiith/vision-sae/pytorch/default/1/vision_sae_final.pt"

# Pre-built dataset directories (created by the baseline notebook task_2).
# If these don't exist yet, the DATA ACQUISITION cell below will re-build them.
DATA_DIR         = Path("/kaggle/working/data")
PROBE_DIR        = Path("/kaggle/working/probe_images")
MANIFEST_PATH    = DATA_DIR / "dataset_manifest.json"

# ── TopK SAE hyper-parameters (must match the saved checkpoint) ──────────
D_MODEL  = 1024
D_SAE    = 1024 * 32   # 32,768
SAE_K    = 32

# ── Feature discovery ────────────────────────────────────────────────────
N_FEATURES_TO_ABLATE = 5   # top-N zebra SAE features to zero out

random.seed(42)
print(f"Model : {MODEL_ID}")
print(f"SAE   : {SAE_PATH}")
print(f"CUDA  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
# ============================================================
# STEP 1: HuggingFace Authentication
# ============================================================
print("=" * 60)
print("STEP 1: HuggingFace Authentication")
print("=" * 60)

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    print("[OK] Retrieved HF_TOKEN from Kaggle secrets.")
except Exception as e:
    print(f"[WARN] Kaggle secrets unavailable ({e}). Trying env variable...")
    token = os.environ.get("HF_TOKEN", None)

if token is None:
    raise RuntimeError("No HF_TOKEN found. Set it as a Kaggle secret or env variable.")

from huggingface_hub import login
login(token=token)
print("[OK] Logged in.\n")

In [ ]:
# ============================================================
# STEP 2: Data Acquisition
# Loads images from disk if the baseline notebook already exported them.
# Otherwise, streams from ImageNet + COCO and saves to disk.
# ============================================================
print("=" * 60)
print("STEP 2: Data Acquisition")
print("=" * 60)

forget_images  = []          # PIL images — zebras
retain_images  = []          # list of (PIL image, label_str)
general_images = []          # COCO — used nowhere in eval but kept for compat

# ── 2a. Try loading from disk (fast path) ────────────────────────────────
if MANIFEST_PATH.exists():
    print("[OK] Found pre-built data/ directory from baseline notebook. Loading from disk...")
    with open(MANIFEST_PATH) as f:
        manifest = json.load(f)

    for i in range(manifest["forget_count"]):
        p = DATA_DIR / "forget" / f"forget_{i}.png"
        if p.exists():
            forget_images.append(Image.open(p).convert("RGB"))

    retain_labels = manifest["retain_labels"]
    for i, lbl in enumerate(retain_labels):
        p = DATA_DIR / "retain" / f"retain_{i}.png"
        if p.exists():
            retain_images.append((Image.open(p).convert("RGB"), lbl))

    print(f"[OK] Loaded {len(forget_images)} forget images and {len(retain_images)} retain images from disk.")

else:
    # ── 2b. Stream from ImageNet (slow path, only if disk data missing) ──
    print("[...] data/ not found — streaming from ImageNet validation...")
    from datasets import load_dataset

    imagenet_val = load_dataset("ILSVRC/imagenet-1k", split="validation", streaming=True)
    class_names  = imagenet_val.features["label"].names
    wnid_to_lbl  = {name: idx for idx, name in enumerate(class_names)}

    zebra_label  = wnid_to_lbl["zebra"]
    horse_label  = wnid_to_lbl["sorrel"]
    donkey_label = wnid_to_lbl["ox"]

    z_count = h_count = d_count = 0
    processed = 0
    it = iter(imagenet_val)

    while True:
        try:
            sample   = next(it)
            processed += 1
            lbl = sample["label"]
            img = sample["image"].convert("RGB")

            if lbl == zebra_label  and z_count < 50:
                forget_images.append(img); z_count += 1
            elif lbl == horse_label  and h_count < 50:
                retain_images.append((img, "horse")); h_count += 1
            elif lbl == donkey_label and d_count < 50:
                retain_images.append((img, "donkey")); d_count += 1

            if z_count >= 50 and h_count >= 50 and d_count >= 50:
                print("[OK] All images collected."); break

        except StopIteration:
            print("[WARN] End of dataset reached."); break
        except Exception as e:
            print(f"[WARN] Network error: {e}. Reconnecting...")
            time.sleep(2)
            it = iter(imagenet_val.skip(processed))

    # Save for reuse
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    (DATA_DIR / "forget").mkdir(exist_ok=True)
    (DATA_DIR / "retain").mkdir(exist_ok=True)
    for i, img in enumerate(forget_images):
        img.save(DATA_DIR / "forget" / f"forget_{i}.png")
    for i, (img, lbl) in enumerate(retain_images):
        img.save(DATA_DIR / "retain" / f"retain_{i}.png")
    print(f"[OK] Saved {len(forget_images)} forget + {len(retain_images)} retain images to disk.")

    del imagenet_val
    gc.collect()

print(f"\nForget set : {len(forget_images)} zebra images")
print(f"Retain set : {len(retain_images)} horse/donkey images")

In [ ]:
# ============================================================
# STEP 3: Build VQA Pairs
# ============================================================
print("=" * 60)
print("STEP 3: Build VQA Pairs")
print("=" * 60)

forget_vqa = [
    {
        "image_id": f"forget_{i}",
        "image":    img,
        "qa_pairs": [
            {"question": "What animal is shown in this image?",           "ground_truth": "zebra"},
            {"question": "Describe the pattern/texture of the animal.",   "ground_truth": "black and white stripes"},
            {"question": "Is this a zebra or a horse?",                   "ground_truth": "zebra"},
        ],
    }
    for i, img in enumerate(forget_images)
]

retain_vqa = [
    {
        "image_id": f"retain_{i}",
        "image":    img,
        "qa_pairs": [
            {"question": "What animal is shown in this image?",           "ground_truth": animal},
            {"question": "Describe the pattern/texture of the animal.",   "ground_truth": "brown/plain coat"},
            {"question": "Is this a zebra or a horse or a donkey?",       "ground_truth": animal},
        ],
    }
    for i, (img, animal) in enumerate(retain_images)
]

print(f"Forget VQA entries : {len(forget_vqa)}  ({len(forget_vqa)*3} QA pairs)")
print(f"Retain VQA entries : {len(retain_vqa)}  ({len(retain_vqa)*3} QA pairs)")

In [ ]:
# ============================================================
# STEP 4: Build Probe Sets
# ============================================================
print("=" * 60)
print("STEP 4: Build Probe Sets")
print("=" * 60)

# ── 4a. Adversarial image probes ─────────────────────────────────────────
(PROBE_DIR / "original").mkdir(parents=True, exist_ok=True)
(PROBE_DIR / "blur").mkdir(exist_ok=True)
(PROBE_DIR / "grey").mkdir(exist_ok=True)

adversarial_probe = []
for i in range(min(50, len(forget_images))):
    orig = forget_images[i]
    blurred    = orig.filter(ImageFilter.GaussianBlur(radius=15))
    greyscale  = orig.convert("L").convert("RGB")

    orig.save(     PROBE_DIR / "original" / f"probe_{i}.png")
    blurred.save(  PROBE_DIR / "blur"     / f"probe_{i}.png")
    greyscale.save(PROBE_DIR / "grey"     / f"probe_{i}.png")

    adversarial_probe.append({
        "image_id":    f"adv_probe_{i}",
        "blurred":     blurred,
        "greyscale":   greyscale,
        "question":    "What animal is shown in this image?",
        "ground_truth": "zebra",
    })

# ── 4b. Text-only probes ─────────────────────────────────────────────────
text_only_probes = [
    {"question": "What does a zebra look like?",                                  "expected_keywords": ["stripes", "black and white"]},
    {"question": "What color pattern do zebras have?",                            "expected_keywords": ["stripes", "black and white"]},
    {"question": "Describe the appearance of a zebra.",                           "expected_keywords": ["stripes", "black and white"]},
    {"question": "How can you tell a zebra apart from a horse?",                  "expected_keywords": ["stripes", "black and white"]},
    {"question": "What makes zebras unique compared to other equines?",           "expected_keywords": ["stripes", "black and white"]},
    {"question": "What is the most distinctive feature of a zebra?",              "expected_keywords": ["stripes", "black and white"]},
    {"question": "If I saw a striped horse-like animal, what would it be?",       "expected_keywords": ["stripes", "black and white", "zebra"]},
    {"question": "What kind of markings does a zebra have?",                      "expected_keywords": ["stripes", "black and white"]},
    {"question": "Are zebras black with white stripes or white with black stripes?", "expected_keywords": ["stripes", "black and white"]},
    {"question": "What is a zebra's coat like?",                                  "expected_keywords": ["stripes", "black and white"]},
    {"question": "Do zebras have spots or stripes?",                              "expected_keywords": ["stripes", "black and white"]},
    {"question": "Name an animal with black and white stripes.",                  "expected_keywords": ["stripes", "black and white", "zebra"]},
    {"question": "What pattern covers a zebra's body?",                           "expected_keywords": ["stripes", "black and white"]},
    {"question": "Describe a zebra to someone who has never seen one.",           "expected_keywords": ["stripes", "black and white"]},
    {"question": "What animal has black and white stripes and lives in Africa?",  "expected_keywords": ["stripes", "black and white", "zebra"]},
    {"question": "What colors are a zebra?",                                      "expected_keywords": ["stripes", "black and white"]},
    {"question": "How would you describe a zebra's fur?",                         "expected_keywords": ["stripes", "black and white"]},
    {"question": "What is distinctive about a zebra's appearance?",               "expected_keywords": ["stripes", "black and white"]},
    {"question": "Which animal is known for its black and white stripes?",        "expected_keywords": ["stripes", "black and white", "zebra"]},
    {"question": "What does a zebra's coat look like?",                           "expected_keywords": ["stripes", "black and white"]},
    {"question": "What patterns does a zebra have on its body?",                  "expected_keywords": ["stripes", "black and white"]},
    {"question": "How is a zebra different from a horse visually?",               "expected_keywords": ["stripes", "black and white"]},
    {"question": "What type of coat does a zebra have?",                          "expected_keywords": ["stripes", "black and white"]},
    {"question": "Can you describe the coloring of a zebra?",                     "expected_keywords": ["stripes", "black and white"]},
    {"question": "What is the typical body pattern of a zebra?",                  "expected_keywords": ["stripes", "black and white"]},
]

print(f"Adversarial probes : {len(adversarial_probe)} images (blur + greyscale treatments)")
print(f"Text-only probes   : {len(text_only_probes)} questions")

In [ ]:
# ============================================================
# STEP 5: TopK SAE Architecture (must match saved checkpoint)
# ============================================================
class TopKSAE(nn.Module):
    """
    TopK Sparse Autoencoder — matches train_vision_sae.py exactly.
    W_enc: (d_sae, d_model)   W_dec: (d_model, d_sae)
    """
    def __init__(self, d_model: int, d_sae: int, k: int):
        super().__init__()
        self.d_model = d_model
        self.d_sae   = d_sae
        self.k       = k

        self.W_enc = nn.Parameter(torch.empty(d_sae, d_model))
        self.b_enc = nn.Parameter(torch.zeros(d_sae))
        self.W_dec = nn.Parameter(torch.empty(d_model, d_sae))
        self.b_dec = nn.Parameter(torch.zeros(d_model))

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        x_centered = x - self.b_dec
        pre_acts   = F.linear(x_centered, self.W_enc, self.b_enc)
        topk_vals, topk_idx = torch.topk(pre_acts, self.k, dim=-1)
        h = torch.zeros_like(pre_acts)
        h.scatter_(-1, topk_idx, F.relu(topk_vals))
        return h

    def decode(self, h: torch.Tensor) -> torch.Tensor:
        return F.linear(h, self.W_dec, self.b_dec)

    def forward(self, x: torch.Tensor):
        h     = self.encode(x)
        x_hat = self.decode(h)
        return x_hat

print("[OK] TopKSAE class defined.")

In [ ]:
# ============================================================
# STEP 6: Load LLaVA-Next Model (4-bit, fits on single T4)
# ============================================================
print("=" * 60)
print("STEP 6: Load LLaVA-Next Model (4-bit NF4)")
print("=" * 60)

from transformers import (
    LlavaNextProcessor,
    LlavaNextForConditionalGeneration,
    BitsAndBytesConfig,
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,       # extra VRAM saving
)

print("[...] Loading processor...")
processor = LlavaNextProcessor.from_pretrained(MODEL_ID)

print("[...] Loading model (4-bit)...")
model = LlavaNextForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)
model.eval()

# Locate vision tower (handles slight HF version differences)
if hasattr(model, 'vision_tower'):
    vision_tower = model.vision_tower
elif hasattr(model, 'model') and hasattr(model.model, 'vision_tower'):
    vision_tower = model.model.vision_tower
else:
    raise AttributeError("Cannot locate vision_tower in model.")

VISION_DEVICE = next(vision_tower.parameters()).device
print(f"[OK] Model loaded. Vision tower on: {VISION_DEVICE}")

In [ ]:
# ============================================================
# STEP 7: Load SAE
# ============================================================
print("=" * 60)
print("STEP 7: Load TopK SAE")
print("=" * 60)

# SAE lives in fp16 on the same device as the vision tower to avoid
# cross-device copies during the forward hook.
sae = TopKSAE(d_model=D_MODEL, d_sae=D_SAE, k=SAE_K)
sae.load_state_dict(torch.load(SAE_PATH, map_location="cpu"))
sae = sae.to(VISION_DEVICE).to(torch.float16)
sae.eval()

total_params = sum(p.numel() for p in sae.parameters())
print(f"[OK] SAE loaded — {D_MODEL} → {D_SAE} (k={SAE_K}), params={total_params:,}")

In [ ]:
# ============================================================
# STEP 8: Feature Discovery
# Find the SAE latents most activated by zebra images.
# Processing one image at a time → no VRAM spike.
# ============================================================
print("=" * 60)
print("STEP 8: Zebra Feature Discovery")
print("=" * 60)

feature_activations = torch.zeros(D_SAE, device=VISION_DEVICE, dtype=torch.float32)

print(f"[...] Running discovery on {len(forget_images)} zebra images...")

with torch.no_grad():
    for idx, img in enumerate(tqdm(forget_images, desc="Feature discovery")):

        # LLaVA-Next uses a conversation template for best results
        conversation = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": "What is in this image?"},
                ],
            }
        ]
        prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
        inputs = processor(images=img, text=prompt, return_tensors="pt").to(VISION_DEVICE)

        pixel_values = inputs.pixel_values.to(torch.float16)

        # LLaVA-Next tiles images into multiple patches: (1, num_tiles, C, H, W)
        if pixel_values.dim() == 5:
            B, num_tiles, C, H, W = pixel_values.shape
            flat_pixels = pixel_values.view(B * num_tiles, C, H, W)
        else:
            flat_pixels = pixel_values

        vision_outputs = vision_tower(flat_pixels, output_hidden_states=True)

        # Use the FINAL hidden state (layer -1) — same layer the hook will intercept
        hidden = vision_outputs.hidden_states[-1].to(torch.float16)  # (tiles, tokens, 1024)

        acts = sae.encode(hidden)                  # (tiles, tokens, D_SAE)
        feature_activations += acts.sum(dim=(0, 1)).float()  # accumulate in fp32

        del inputs, pixel_values, flat_pixels, vision_outputs, hidden, acts
        torch.cuda.empty_cache()

# Top-N zebra features
FEATURES_TO_ABLATE = torch.topk(feature_activations, N_FEATURES_TO_ABLATE).indices.tolist()
print(f"\n[OK] Top {N_FEATURES_TO_ABLATE} zebra SAE features to ablate: {FEATURES_TO_ABLATE}")
del feature_activations
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# ============================================================
# STEP 9: Install Zero-Ablation Hook
# ============================================================
print("=" * 60)
print("STEP 9: Install SAE Zero-Ablation Hook")
print("=" * 60)

def make_zero_ablation_hook(features_to_ablate: list, sae: TopKSAE):
    """
    Returns a forward hook for a ViT transformer layer.

    On every forward pass the hook:
      1. Extracts the hidden-state tensor from the layer output.
      2. Encodes it through the SAE.
      3. Zeros the target zebra features.
      4. Decodes back to activation space.
      5. Returns the modified tensor so downstream layers see no zebra signal.
    """
    features_tensor = torch.tensor(features_to_ablate, device=VISION_DEVICE, dtype=torch.long)

    def hook_fn(module, input_args, output):
        # Unpack hidden states — output can be a Tensor or a tuple
        if isinstance(output, torch.Tensor):
            hidden = output
            is_tuple = False
        else:
            hidden = output[0]
            is_tuple = True

        orig_device = hidden.device
        orig_dtype  = hidden.dtype

        # Move to SAE device in fp16
        h = hidden.to(VISION_DEVICE).to(torch.float16)

        with torch.no_grad():
            # 1. Encode → SAE latent space
            acts = sae.encode(h)          # (..., D_SAE)

            # 2. Zero ablate target features
            acts[..., features_tensor] = 0.0

            # 3. Decode back to activation space
            reconstructed = sae.decode(acts)

        # Restore original device + dtype
        reconstructed = reconstructed.to(orig_device).to(orig_dtype)

        if is_tuple:
            return (reconstructed,) + output[1:]
        return reconstructed

    return hook_fn


# Hook the LAST encoder layer of the vision tower
# (same layer where activations were extracted during SAE training)
last_layer = vision_tower.vision_model.encoder.layers[-1]
hook_handle = last_layer.register_forward_hook(
    make_zero_ablation_hook(FEATURES_TO_ABLATE, sae)
)

print(f"[OK] Zero-ablation hook installed on vision_tower.vision_model.encoder.layers[-1]")
print(f"     Ablating features: {FEATURES_TO_ABLATE}")

In [ ]:
# ============================================================
# STEP 10: Inference Helper
# ============================================================

def ask_vlm(image, question: str, max_new_tokens: int = 100) -> str:
    """
    Run LLaVA-Next inference. The zero-ablation hook fires automatically
    during the forward pass when image is not None.
    If image is None, runs text-only (LL probe — hook still fires but
    vision inputs are a dummy; LL measures LLM's language knowledge).
    """
    if image is not None:
        conversation = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": question},
                ],
            }
        ]
        prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
        inputs = processor(images=image, text=prompt, return_tensors="pt").to(model.device)
    else:
        # Text-only: no image token — measures the LLM's pure language knowledge
        conversation = [
            {"role": "user", "content": [{"type": "text", "text": question}]}
        ]
        prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
        inputs = processor(text=prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
        )

    generated = processor.decode(
        output_ids[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )
    return generated.strip()


# Quick sanity check — the model should now be uncertain / confused about zebras
print("[Sanity check] asking about a zebra image WITH zero-ablation hook active...")
test_ans = ask_vlm(forget_images[0], "What animal is shown in this image?", max_new_tokens=30)
print(f"  Response: {test_ans}")

In [ ]:
# ============================================================
# STEP 11a: Evaluate Forget Set (FA)
# FA = % of forget-set answers that still mention "zebra".
# Lower is better for unlearning.
# ============================================================
print("=" * 60)
print("STEP 11a: Forget Accuracy (FA)")
print("=" * 60)

forget_responses = []
forget_correct   = 0
forget_total     = 0

for entry in tqdm(forget_vqa, desc="Forget Set"):
    for qa in entry["qa_pairs"]:
        answer     = ask_vlm(entry["image"], qa["question"])
        is_correct = "zebra" in answer.lower()
        forget_correct += int(is_correct)
        forget_total   += 1
        forget_responses.append({
            "image_id":     entry["image_id"],
            "question":     qa["question"],
            "ground_truth": qa["ground_truth"],
            "model_answer": answer,
            "correct":      is_correct,
        })

FA = (forget_correct / forget_total) * 100 if forget_total else 0
print(f"\n✅ Forget Accuracy (FA): {FA:.2f}%  ({forget_correct}/{forget_total})")
print("   (Lower = better unlearning)")

In [ ]:
# ============================================================
# STEP 11b: Evaluate Retain Set (RA)
# RA = % of retain-set answers that are correct.
# Higher is better — we want to preserve general capability.
# ============================================================
print("=" * 60)
print("STEP 11b: Retain Accuracy (RA)")
print("=" * 60)

def normalize(text: str) -> str:
    return text.lower().strip()

def is_match(gt: str, answer: str) -> bool:
    gt     = normalize(gt)
    answer = normalize(answer)
    if gt in ["horse", "donkey", "zebra"]:
        keywords = {
            "horse":  ["horse", "stallion", "mare"],
            "donkey": ["donkey", "ox"],
            "zebra":  ["zebra"],
        }
        return any(w in answer for w in keywords[gt])
    elif gt == "brown/plain coat":
        return any(w in answer for w in ["brown", "plain"])
    elif gt == "black and white stripes":
        return any(w in answer for w in ["black", "white", "stripe"])
    return gt in answer


retain_responses = []
retain_correct   = 0
retain_total     = 0

for entry in tqdm(retain_vqa, desc="Retain Set"):
    for qa in entry["qa_pairs"]:
        answer     = ask_vlm(entry["image"], qa["question"])
        is_correct = is_match(qa["ground_truth"], answer)
        retain_correct += int(is_correct)
        retain_total   += 1
        retain_responses.append({
            "image_id":     entry["image_id"],
            "question":     qa["question"],
            "ground_truth": qa["ground_truth"],
            "model_answer": answer,
            "correct":      is_correct,
        })

RA = (retain_correct / retain_total) * 100 if retain_total else 0
print(f"\n✅ Retain Accuracy (RA): {RA:.2f}%  ({retain_correct}/{retain_total})")
print("   (Higher = better; should stay close to baseline ~71%)")

In [ ]:
# ============================================================
# STEP 11c: Language Leakage (LL) — Text-only probes
# LL = % of text-only zebra questions answered with stripe keywords.
# This measures LLM-level (not vision-level) knowledge retention.
# Zero-ablation operates on vision features, so LL should be unchanged.
# ============================================================
print("=" * 60)
print("STEP 11c: Language Leakage (LL) — Text-only probes")
print("=" * 60)

text_probe_responses = []
leak_count = 0

for probe in tqdm(text_only_probes, desc="Text-Only Probes"):
    answer = ask_vlm(None, probe["question"])
    a_low  = answer.lower()
    leaked = ("stripes" in a_low) or ("black and white" in a_low)
    leak_count += int(leaked)
    text_probe_responses.append({
        "question":          probe["question"],
        "expected_keywords": probe["expected_keywords"],
        "model_answer":      answer,
        "leakage_detected":  leaked,
    })

LL = (leak_count / len(text_only_probes)) * 100
print(f"\n✅ Language Leakage (LL): {LL:.2f}%  ({leak_count}/{len(text_only_probes)})")
print("   (Expected: similar to baseline ~96% — vision hook doesn't affect text knowledge)")

In [ ]:
# ============================================================
# STEP 11d: Adversarial Robustness (AR)
# AR = % of blurred/greyscale zebra answers that still identify "zebra".
# Lower is better for unlearning.
# ============================================================
print("=" * 60)
print("STEP 11d: Adversarial Robustness (AR) — Blurred + Greyscale probes")
print("=" * 60)

adv_responses = []
adv_correct   = 0
adv_total     = 0

for probe in tqdm(adversarial_probe, desc="Adversarial Probes"):
    for treatment_name, treatment_img in [
        ("blurred",   probe["blurred"]),
        ("greyscale", probe["greyscale"]),
    ]:
        answer     = ask_vlm(treatment_img, probe["question"])
        is_correct = "zebra" in answer.lower()
        adv_correct += int(is_correct)
        adv_total   += 1
        adv_responses.append({
            "image_id":     probe["image_id"],
            "treatment":    treatment_name,
            "question":     probe["question"],
            "ground_truth": probe["ground_truth"],
            "model_answer": answer,
            "correct":      is_correct,
        })

AR = (adv_correct / adv_total) * 100 if adv_total else 0
print(f"\n✅ Adversarial Robustness (AR): {AR:.2f}%  ({adv_correct}/{adv_total})")
print("   (Lower = better unlearning)")

In [ ]:
# ============================================================
# STEP 12: Remove Hook + Print Summary
# ============================================================
hook_handle.remove()
print("[OK] Forward hook removed.\n")

# Baseline numbers from task_2 for easy comparison
BASELINE = {"FA": 98.67, "RA": 71.33, "LL": 96.00, "AR": 50.00}

print("=" * 65)
print("  SAE ZERO-ABLATION — METRICS SUMMARY")
print("=" * 65)
print(f"  {'Metric':<35} {'Baseline':>9}  {'Ablated':>9}  {'Delta':>8}")
print(f"  {'-'*35} {'-'*9}  {'-'*9}  {'-'*8}")

results_now = {"FA": FA, "RA": RA, "LL": LL, "AR": AR}
labels = {
    "FA": "FA  (Forget Accuracy %)",
    "RA": "RA  (Retain Accuracy %)",
    "LL": "LL  (Language Leakage %)",
    "AR": "AR  (Adversarial Robustness %)",
}
for key, label in labels.items():
    base  = BASELINE[key]
    now   = results_now[key]
    delta = now - base
    print(f"  {label:<35} {base:>8.2f}%  {now:>8.2f}%  {delta:>+7.2f}%")

print("=" * 65)
print("\nInterpretation:")
print("  FA ↓  → model less likely to identify zebras (good for unlearning)")
print("  RA ≈  → general capability preserved")
print("  LL ≈  → LLM text knowledge unchanged (expected; vision-only intervention)")
print("  AR ↓  → model less robust to adversarial zebra variants")

In [ ]:
# ============================================================
# STEP 13: Save All Results
# ============================================================
all_results = {
    "method":          "SAE Zero-Ablation",
    "model":           MODEL_ID,
    "sae_path":        SAE_PATH,
    "features_ablated": FEATURES_TO_ABLATE,
    "n_features":      N_FEATURES_TO_ABLATE,
    "quantization":    "4bit-nf4",
    "metrics": {
        "FA (Forget Accuracy %)":        round(FA, 2),
        "RA (Retain Accuracy %)":        round(RA, 2),
        "LL (Language Leakage %)":       round(LL, 2),
        "AR (Adversarial Robustness %)": round(AR, 2),
    },
    "baseline_metrics": BASELINE,
    "forget_responses":          forget_responses,
    "retain_responses":          retain_responses,
    "text_probe_responses":      text_probe_responses,
    "adversarial_probe_responses": adv_responses,
}

out_path = "/kaggle/working/sae_zero_ablation_results.json"
with open(out_path, "w") as f:
    json.dump(all_results, f, indent=2)

print(f"✅ Results saved to {out_path}")
print("\nFor the submission sheet — SAE Zero-Ablation row:")
print(f"  FA = {FA:.2f}%")
print(f"  RA = {RA:.2f}%")
print(f"  LL = {LL:.2f}%")
print(f"  AR = {AR:.2f}%")